## `stat`, `readlink`, and Symlinks vs. Hardlinks

**Support context:** "Broken symlink" is one of the most common real-world Linux tickets — an app can't find a file that _looks_ like it's there in `ls`, but points nowhere. You need to be able to spot and diagnose this instantly.

### `stat` — full metadata, beyond what `ls -l` shows

In [ ]:
dennie@HP:~/linux-tut$ stat /etc/passwd
  File: /etc/passwd
  Size: 1426            Blocks: 8          IO Block: 4096   regular file
Device: 8,48    Inode: 49343       Links: 1
Access: (0644/-rw-r--r--)  Uid: (    0/    root)   Gid: (    0/    root)
Access: 2026-08-24 15:26:03.188713900 +0000
Modify: 2026-08-21 21:33:59.266788401 +0000
Change: 2026-08-21 21:33:59.267390275 +0000
 Birth: 2026-08-21 21:33:59.263304930 +0000

**READ THE OUTPUT**: you'll see `Access`, `Modify`, `Change` timestamps (three different things — we'll unpack the difference), the **inode number**, number of hard links, and permissions — all in one shot.
- **Access** (atime) — last time the file's _content_ was read. Yours shows Aug 24 15:26 — later than Modify, meaning something (a `cat`, `grep`, or similar) read it recently without changing it, which is exactly the kind of read-only access that bumps atime alone.
- **Modify** (mtime) — last time the file's _content_ changed. This is what `ls -l` shows you by default, and what `find -mtime` filters on.
- **Change** (ctime) — last time the file's **metadata** changed (permissions, ownership, or content — anything about the inode itself). Often confused with "creation time" — it isn't.
- **Birth** — actual creation time. Not tracked on every filesystem/kernel combo, but yours shows one.
- **Links: 1** — only one hardlink exists for `/etc/passwd`: itself. We'll see this number change in a second.

> `ls -l` only shows you a summary; `stat` shows you everything the filesystem actually tracks about that file.

### `readlink` — **follow a symlink to its real target**

`readlink` doesn't read filenames from **stdin** — it expects them as **command-line arguments**, the same way `ls somefile.txt` needs `somefile.txt` typed right after it, not fed in through a pipe. 💡 Why: commands split into two rough categories:

- **Stdin-readers** (`grep`, `sort`, `uniq`, `awk`, `sed`, `cat`, `wc`) — these are built to receive a stream of _data_ on stdin and process it line by line. Piping into them works naturally.
- **Argument-takers** (`readlink`, `ls`, `chmod`, `rm`, `cp`, `stat`, `file`) — these expect the _filename itself_ as an argument on the command line. Piping a filename into their stdin does nothing useful — they're not looking there at all.

🔁 **Support rule:** when a pipe "does nothing" or errors with something like `missing operand`, ask "does this command actually read stdin, or does it want arguments?" That's exactly what `xargs` exists to bridge — remember from the Pillar 4 lesson: _"pass piped output as arguments to another command."_ This is precisely the situation it was built for.

In [ ]:
dennie@HP:~/linux-tut$ readlink /bin
usr/bin

In [ ]:
dennie@HP:~/linux-tut$ readlink -f /bin/ls
/usr/bin/ls

**`readlink /bin` → `usr/bin`** (no leading slash) — the symlink's stored target is exactly what was set: a **relative** path, not absolute. It works because it's relative to `/` itself (the symlink's own location), so `usr/bin` from `/` resolves correctly. `readlink -f` doesn't care either way — it fully resolves regardless of relative or absolute storage.

**FLAGS:** plain `readlink` shows the immediate target (one hop). `-f` resolves the **entire chain** down to the final real file, even through multiple nested symlinks.

### Symlinks vs. Hardlinks — the core concept

We already saw this live: `/bin -> usr/bin`.
> A **symlink** is a separate, tiny file that just contains a _path_ pointing somewhere else — like a shortcut.

> A **hardlink** is a second directory entry pointing at the exact same **inode** (the actual data on disk) — not a pointer to a path, but literally another name for the same file.

> Symbolic links, also known as **soft links**, are created using the `-s` option with the `ln` command. Symbolic links are **pointers to the original file** or **directory**, and they contain the **path to the target**.

Unlike hard links, symbolic links can point to files or directories **across file system boundaries**.

To create a symbolic link, run the following command:

```bash
ln -s original_file symbolic_link_name
```

```bash
$ ln -s file1.txt file1_symlink.txt
```

The main difference between hard links and symbolic links is that hard links are **tightly coupled with the original file**, while symbolic links are **more flexible** and can point to files or directories across the file system.

Now, let's create a hard link to `file1.txt` using the `ln` command:
```bash
ln file1.txt file1_hardlink.txt
```
The `ln` command **without any options** creates a **hard link**. This means that `file1_hardlink.txt` and `file1.txt` now share the **same inode** (file metadata) and **physical storage location**.
> Any changes made to one file will be reflected in the other.

In [ ]:
dennie@HP:~/linux-tut$ ls -li /bin/ls /usr/bin/ls
1799 -rwxr-xr-x 1 root root 142312 Jun 22  2025 /bin/ls
1799 -rwxr-xr-x 1 root root 142312 Jun 22  2025 /usr/bin/ls

**`ls -li /bin/ls /usr/bin/ls` — both inode `1799`** — careful here, this looks like a hardlink demo but it _isn't_ one. There's only **one** real file (`/usr/bin/ls`). `/bin` is a symlink to `usr/bin`, so `/bin/ls` isn't a second file with its own directory entry — it's the _same_ directory entry, just reached by first hopping through the `/bin` symlink. One file, two paths to get there. To see a genuine hardlink, we need to build one — that's next.

**WHAT TO LOOK FOR:** the leftmost number `ls -li` prints is the **inode**. Try comparing that against a symlink like `/bin` (a directory symlink, so this exact command won't show it, but conceptually): a symlink has its _own_ inode (it's a real, separate file containing a path string), while a hardlink shares the _same_ inode number as the file it's linked to.

**TICKET SCENARIO:** _"App says config file not found, but `ls` clearly shows it there."_

In [ ]:
ls -l /etc/some-symlinked-config

> If the entry starts with `l` (symlink) and `ls -l` shows it in a different color (often red/blinking in most terminals) pointing to a path that no longer exists — that's a **broken symlink**: the shortcut exists, but its target was deleted or moved. `readlink -f` on it will confirm by failing to resolve.

---

### Hands-On: Build a Real Hardlink and Symlink, Then Break One

This is the exercise that makes the difference click permanently. Run each block, and look at what I tell you to check before moving to the next.

**Step 1 — create the original and both link types:**
```bash
ln /tmp/original.txt /tmp/hardlink.txt
# TARGET (what it points TO)   LINK_NAME (the symlink file itself, created)
```
- `/tmp/hardlink.txt` is the link file
- `/tmp/original.txt` path string stored inside it

In [ ]:
touch /tmp/original.txt
ln /tmp/original.txt /tmp/hardlink.txt
ln -s /tmp/original.txt /tmp/symlink.txt

**Step 2 — compare them:**

In [ ]:
dennie@HP:~/linux-tut$ ls -li /tmp/original.txt /tmp/hardlink.txt /tmp/symlink.txt
41021 -rw-r--r-- 2 dennie dennie  0 Aug 25 04:20 /tmp/hardlink.txt
41021 -rw-r--r-- 2 dennie dennie  0 Aug 25 04:20 /tmp/original.txt
45467 lrwxrwxrwx 1 dennie dennie 17 Aug 25 04:21 /tmp/symlink.txt -> /tmp/original.txt

**WHAT TO LOOK FOR:** `original.txt` and `hardlink.txt` share the **same inode number** and both show `Links: 2` in `stat` — they're genuinely two names for one underlying file now. `symlink.txt` has its **own, different, tiny inode**, shows as `lrwxrwxrwx` (note the leading `l`), and its listed size is just the length of the path string it stores — `-> /tmp/original.txt`.

In [ ]:
dennie@HP:~/linux-tut$ stat /tmp/original.txt | grep Links
Device: 8,48    Inode: 41021       Links: 2
dennie@HP:~/linux-tut$ stat /tmp/hardlink.txt | grep Links
Device: 8,48    Inode: 41021       Links: 2

**Step 3 — the actual point: delete the "original" and see what survives:**

In [ ]:
dennie@HP:~/linux-tut$ rm /tmp/original.txt
dennie@HP:~/linux-tut$ cat /tmp/hardlink.txt && echo 'hardlink: still readable'
hardlink: still readable

In [ ]:
dennie@HP:~/linux-tut$ cat /tmp/symlink.txt
cat: /tmp/symlink.txt: No such file or directory

In [ ]:
dennie@HP:~/linux-tut$ ls -l /tmp/symlink.txt
lrwxrwxrwx 1 dennie dennie 17 Aug 25 04:21 /tmp/symlink.txt -> /tmp/original.txt

**READ THE OUTPUT:** the hardlink is completely unaffected — the actual data isn't deleted until the _last_ hardlink pointing at that inode is removed (its link count just drops from 2 back to 1). The symlink, though, now points at a path that no longer exists — `cat` fails with `No such file or directory`, but `ls -l` still shows the symlink entry itself (in red, typically), pointing at a target that's gone.
> **That's a broken symlink** — the exact thing a "file not found but `ls` shows it" ticket looks like.

In [ ]:
rm /tmp/hardlink.txt /tmp/symlink.txt

(cleanup — nothing left behind)

Run all three steps and paste what you see, especially the `Links:` counts and the final broken-symlink behavior — then we move to `wc`, `sort`, `uniq`, `head`/`tail -f`, `less`, `diff`, `touch`.

Confirmed exactly as expected — the hardlink didn't care that `original.txt` was deleted (still `Links: 1` remaining, data intact), while the symlink became a dangling pointer. That distinction is now locked in.

---

#### Rest of Pillar 1: `wc`, `sort`, `uniq`, `head`/`tail -f`, `less`, `diff`, `touch`

**Support context:** these are the building blocks of every **log-analysis pipeline** you'll ever run.
> Individually simple; combined, they answer "what's actually going on in this log file" faster than opening it in an editor ever could.

### `wc` — word/line/character count

In [ ]:
wc -l /etc/passwd
28 /etc/passwd

**FLAGS:** `-l` lines, `-w` words, `-c` bytes, `-m` characters (differs from `-c` with multi-byte/unicode text). `-l` is by far the one you'll use most.

**TICKET SCENARIO:** _"How many users are on this box?"_ → `wc -l /etc/passwd` (roughly — one line per account). _"How many failed logins today?"_ → pipe a `grep` result into `wc -l` to get a count instead of a wall of text.

---

### `sort` + `uniq` — the log-analysis workhorse combo

`uniq` only collapses **adjacent** duplicate lines — so you almost always `sort` first.

In [ ]:
sort /tmp/example.txt | uniq -c | sort -rn

**FLAGS:**

|Flag|Effect|
|---|---|
|`sort -n`|numeric sort (not alphabetical — `"10"` sorts _after_ `"9"`, not before)|
|`sort -r`|reverse order|
|`sort -k2`|sort by the 2nd whitespace-delimited column|
|`uniq -c`|prefix each line with a **count** of how many times it repeated|

**THE classic pipeline — top 10 most common lines in a log:**

In [ ]:
sudo cat /var/log/auth.log | awk '{$1=$2=$3=""; print $0}' | sort | uniq -c | sort -rn | head -10

(We haven't covered `awk` yet — Pillar 4 — so for now just try the simpler version below and we'll build the full one properly later.)

**TRY NOW — a simpler real version you can run today:**

In [ ]:
dennie@HP:~/linux-tut$ sudo grep sudo /var/log/auth.log | wc -l
[sudo] password for dennie: 
30

In [ ]:
dennie@HP:~/linux-tut$ sudo cat /var/log/auth.log | cut -d' ' -f5 | sort | uniq -c | sort -rn | head
     59 session
     30 rules
     13 
     11 loading,
      8 garbage
      6 unable
      6 seat
      6 adding
      5 system
      5 is

**READ THE OUTPUT:** the second command counts how often each distinct "5th word" appears across every auth log line — a rough first pass at "what kind of events dominate this log." `sort -rn` puts the most frequent at the top. This exact `sort | uniq -c | sort -rn` shape is the single most-repeated pattern in log troubleshooting — memorize the shape, not just the example.

---

### `head`, `tail`, `tail -f` — you know `head`/`tail`; here's the critical one

In [ ]:
dennie@HP:~/linux-tut$ tail -f /var/log/syslog
2026-08-25T17:17:01.769849+00:00 HP CRON[1298]: (root) CMD (cd / && run-parts --report /etc/cron.hourly)
2026-08-25T17:17:56.201168+00:00 HP wsl-pro-service[169]: #033[33mWARNING#033[0m Daemon: could not connect to Windows Agent: could not get address: could not read agent port file "/mnt/c/Users/dennie/.ubuntupro/.address": open /mnt/c/Users/dennie/.ubuntupro/.address: no such file or directory
2026-08-25T17:18:59.543698+00:00 HP kernel: WSL (107) ERROR: CheckConnection: getaddrinfo() failed: -5
2026-08-25T17:19:56.262283+00:00 HP wsl-pro-service[169]: message repeated 2 times: [ #033[33mWARNING#033[0m Daemon: could not connect to Windows Agent: could not get address: could not read agent port file "/mnt/c/Users/dennie/.ubuntupro/.address": open /mnt/c/Users/dennie/.ubuntupro/.address: no such file or directory]
2026-08-25T17:19:56.262555+00:00 HP wsl-pro-service[169]: #033[33mWARNING#033[0m Exiting after <nil>: check if the Windows agent is installed and running.
2026-08-25T17:19:56.265693+00:00 HP systemd[1]: wsl-pro.service: Deactivated successfully.
2026-08-25T17:19:56.266016+00:00 HP systemd[1]: wsl-pro.service: Consumed 1.095s CPU time.
2026-08-25T17:23:49.658820+00:00 HP systemd[1]: Starting systemd-tmpfiles-clean.service - Cleanup of Temporary Directories...
2026-08-25T17:23:49.743864+00:00 HP systemd[1]: systemd-tmpfiles-clean.service: Deactivated successfully.
2026-08-25T17:23:49.744238+00:00 HP systemd[1]: Finished systemd-tmpfiles-clean.service - Cleanup of Temporary Directories.


Leave it running, open a **second** terminal tab, run `logger "test message from tutorial"`:

In [ ]:
dennie@HP:~/linux-tut$ logger "test message from tutorial"

and watch it appear live in the first. `Ctrl+C` to stop following:

In [ ]:
2026-08-25T17:27:07.989789+00:00 HP dennie: test message from tutorial

**PURPOSE (support):** watches a log file **live**, printing new lines as they're written — this is what you run in one terminal while reproducing an issue in another, watching the error happen in real time instead of guessing and re-checking after the fact.

**FLAGS:** `-n 50` show last 50 lines instead of default 10. `-f` follow (stays running until you `Ctrl+C`). `tail -F` (capital) is safer for production — it re-attaches if the log file gets rotated out from under it (a log rotation happening mid-`tail` is a real, WSL-relevant gotcha since Ubuntu rotates `/var/log` on its own schedule).

---

**`less` (and `more`) — paging without flooding your terminal**

In [ ]:
less /var/log/dpkg.log
2026-02-10 00:54:43 startup archives install
2026-02-10 00:54:43 install base-passwd:amd64 <none> 3.6.3build1
2026-02-10 00:54:43 status half-installed base-passwd:amd64 3.6.3build1
...
2026-02-10 00:54:43 status installed base-files:amd64 13ubuntu10
:

**Key bindings inside `less`:** `space`/`f` next page, `b` previous page, `/pattern` search forward, `n` next match, `q` quit. `less` loads incrementally — safe even on huge files, unlike `cat`, which dumps everything at once.

**GOTCHA:** `more` is the older, more limited ancestor (can't scroll backward on all systems). Just use `less` — the name is a joke ("less is more"), not a downgrade.

---

### `diff` — compare two files or configs

In [ ]:
diff /etc/hosts /etc/hosts.bak 2>/dev/null || echo "no backup to compare yet"
no backup to compare yet

**FLAGS:** `-u` unified format (the `+`/`-` style you've seen in git diffs — much more readable). `-q` just says whether files differ, not how.

**TICKET SCENARIO:** _"This config used to work, now it doesn't — what changed?"_ If you (or a deploy pipeline) keep a `.bak` before edits, `diff -u current.conf current.conf.bak` shows exactly what moved, instantly, instead of eyeballing two files side by side.

---

### `touch` — create empty files, or just bump a timestamp

In [ ]:
touch /tmp/newfile.txt
touch /tmp/newfile.txt

In [ ]:
dennie@HP:~/linux-tut$ stat /tmp/newfile.txt | grep Modify
Modify: 2026-08-25 18:04:47.654803076 +0000

command combination variant:

In [ ]:
touch /tmp/newfile.txt && stat /tmp/newfile.txt | grep Modify

**READ THE OUTPUT:** run `touch` on an _existing_ file and nothing about its content changes — only its Modify/Access timestamps jump to now. Useful for triggering things that watch mtime (some file-watchers, some build tools) without editing content.

---